# SkinSight — multi-model evaluation

This notebook scores six published Hugging Face classifiers on the **test split** of
`ahmed-ai/skin-lesions-classification-dataset`.

It does **not** train models and it does **not** invent metrics. Figures and the CSV
are written to the project `results/` folder after a real inference run.

Metrics: accuracy, weighted/macro F1, weighted precision, weighted recall, weighted ROC-AUC (one-vs-rest),
plus a classification report, confusion matrices, and a per-class F1 heatmap.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from transformers import AutoImageProcessor, AutoModelForImageClassification

from app.config import (
    EVAL_BATCH_SIZE,
    EVAL_DATASET_ID,
    EVAL_MODEL_IDS,
    EVAL_MODEL_NAMES,
    EVAL_SPLIT,
    RESULTS_DIR,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("project root:", PROJECT_ROOT)
print("results dir:", RESULTS_DIR)
print("device:", device)

In [ ]:
print("models to evaluate")
for name, model_id in zip(EVAL_MODEL_NAMES, EVAL_MODEL_IDS):
    print(f"  {name:28s} {model_id}")

In [ ]:
print("loading dataset", EVAL_DATASET_ID)
dataset = load_dataset(EVAL_DATASET_ID)
test_split = dataset[EVAL_SPLIT]
class_names = test_split.features["label"].names
num_classes = len(class_names)
print(f"split={EVAL_SPLIT}  n={len(test_split)}  classes={num_classes}")
print(class_names)

In [ ]:
def run_inference(model_id):
    """Return true labels, predicted labels, and class probabilities."""
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForImageClassification.from_pretrained(model_id)
    model.to(device)
    model.eval()

    logits_chunks = []
    labels = []
    total = len(test_split)

    for start in range(0, total, EVAL_BATCH_SIZE):
        end = min(start + EVAL_BATCH_SIZE, total)
        batch = test_split.select(range(start, end))
        images = [image.convert("RGB") for image in batch["image"]]
        encoded = processor(images=images, return_tensors="pt")
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
        logits_chunks.append(outputs.logits.cpu())
        labels.extend(batch["label"])
        if (start // EVAL_BATCH_SIZE) % 10 == 0:
            print(f"    {end}/{total}")

    logits = torch.cat(logits_chunks, dim=0)
    probs = torch.softmax(logits, dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return np.array(labels), preds, probs

In [ ]:
def compute_metrics(y_true, y_pred, probs):
    metrics = {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "F1 (Weighted)": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "F1 (Macro)": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Precision (Weighted)": round(precision_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "Recall (Weighted)": round(recall_score(y_true, y_pred, average="weighted", zero_division=0), 4),
    }
    try:
        y_bin = label_binarize(y_true, classes=list(range(num_classes)))
        auc = roc_auc_score(y_bin, probs, average="weighted", multi_class="ovr")
        metrics["ROC-AUC (Weighted)"] = round(float(auc), 4)
    except Exception:
        metrics["ROC-AUC (Weighted)"] = float("nan")
    return metrics

In [ ]:
all_metrics = {}
all_true = {}
all_pred = {}

for model_id, name in zip(EVAL_MODEL_IDS, EVAL_MODEL_NAMES):
    print("=" * 60)
    print("evaluating", name)
    print(model_id)
    try:
        y_true, y_pred, probs = run_inference(model_id)
        metrics = compute_metrics(y_true, y_pred, probs)
        report = classification_report(
            y_true, y_pred, target_names=class_names, zero_division=0
        )
        all_metrics[name] = metrics
        all_true[name] = y_true
        all_pred[name] = y_pred
        print(metrics)
        print(report)
    except Exception:
        import traceback
        traceback.print_exc()
        all_metrics[name] = {}

In [ ]:
successful = {name: values for name, values in all_metrics.items() if values}
failed = [name for name, values in all_metrics.items() if not values]
print("failed:", failed or "none")

if not successful:
    raise SystemExit("No model produced metrics. Fix errors above before plotting.")

summary = pd.DataFrame(successful).T.sort_values("F1 (Weighted)", ascending=False)
csv_path = RESULTS_DIR / "model_comparison_summary.csv"
summary.to_csv(csv_path)
print(summary.to_string())
print("wrote", csv_path)

In [ ]:
metric_cols = [
    "Accuracy", "F1 (Weighted)", "F1 (Macro)",
    "Precision (Weighted)", "Recall (Weighted)",
]
plot_df = summary[metric_cols].reset_index().rename(columns={"index": "Model"})
x = np.arange(len(plot_df))
width = 0.15
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

fig, ax = plt.subplots(figsize=(15, 6))
for i, col in enumerate(metric_cols):
    values = plot_df[col].astype(float).values
    bars = ax.bar(x + i * width, values, width, label=col, color=colors[i], alpha=0.9)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{value:.3f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
ax.set_xticks(x + width * (len(metric_cols) - 1) / 2)
ax.set_xticklabels(plot_df["Model"], rotation=15, ha="right")
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("SkinSight model comparison (test split)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
bar_path = RESULTS_DIR / "model_comparison_bar.png"
fig.savefig(bar_path, dpi=200)
plt.show()
print("wrote", bar_path)

In [ ]:
def draw_confusion(ax, y_true, y_pred, title):
    matrix = confusion_matrix(y_true, y_pred)
    normalized = matrix.astype(float) / matrix.sum(axis=1, keepdims=True)
    sns.heatmap(
        normalized, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.4, annot_kws={"size": 7},
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

n = len(all_true)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 6))
axes = np.array(axes).reshape(-1)
for i, (name, y_true) in enumerate(all_true.items()):
    draw_confusion(axes[i], y_true, all_pred[name], name)
for j in range(n, len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Normalized confusion matrices", y=1.01)
fig.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrices_all.png"
fig.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.show()
print("wrote", cm_path)

In [ ]:
per_class = {}
for name in all_true:
    report = classification_report(
        all_true[name], all_pred[name],
        target_names=class_names, output_dict=True, zero_division=0,
    )
    per_class[name] = {cls: report[cls]["f1-score"] for cls in class_names}

f1_df = pd.DataFrame(per_class).T
fig, ax = plt.subplots(figsize=(max(12, num_classes * 1.2), len(all_true) * 1.1 + 1))
sns.heatmap(f1_df, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
ax.set_title("Per-class F1")
ax.set_xlabel("Class")
ax.set_ylabel("Model")
plt.xticks(rotation=45)
fig.tight_layout()
heat_path = RESULTS_DIR / "per_class_f1_heatmap.png"
fig.savefig(heat_path, dpi=200, bbox_inches="tight")
plt.show()
print("wrote", heat_path)

## Outputs

If every cell above completed, `results/` now contains:

- `model_comparison_summary.csv`
- `model_comparison_bar.png`
- `confusion_matrices_all.png`
- `per_class_f1_heatmap.png`

Those files are produced by this run. Empty or missing files mean the loop did not finish.